In [11]:
import os
import re
import glob

import numpy as np
import pandas as pd
import networkx as nx
from ase.io import read
from joblib import Parallel, delayed
from ase import __version__ as aseversion
from joblib import __version__ as joblibversion

In [12]:
print("Versions:")
print("numpy:",np.__version__)
print("pandas:",pd.__version__)
print("networkx",nx.__version__)
print("ase:",aseversion)
print("joblib:",joblibversion)

Versions:
numpy: 1.25.2
pandas: 2.2.2
networkx 2.8.8
ase: 3.22.1
joblib: 1.2.0


In [ ]:
def distanceNN(cif_file, Cn):
    """Calculate unique N-N distances for DAH cations with Cn carbons in a CIF structure."""
    if not os.path.exists(cif_file):
        print(f"Warning: {cif_file} does not exist. Skipping.")
        return
    if os.path.getsize(cif_file) == 0:
        print(f"Warning: {cif_file} is empty. Skipping.")
        return
    try:
        structure = read(cif_file)
        distances = []

        # Build a graph of covalently bonded atoms
        G = nx.Graph()
        for atom in structure:
            G.add_node(atom.index, element=atom.symbol)

        for i, atom1 in enumerate(structure):
            for j, atom2 in enumerate(structure):
                if i < j:  # Avoid duplicate checks
                    distance = structure.get_distance(i, j, mic=True)
                    # Use a fixed 1.6 Å cutoff rather than the covalent radii sum
                    if 0 <= distance <= 1.6:
                        G.add_edge(atom1.index, atom2.index)

        # Find connected components (individual molecules)
        molecules = list(nx.connected_components(G))

        # For each DAH molecule (2 nitrogens, Cn carbons), compute the N-N distance
        # by summing displacement vectors along the shortest bonded path
        for molecule in molecules:
            nitrogens = [idx for idx in molecule if structure[idx].symbol == 'N']
            carbons = [idx for idx in molecule if structure[idx].symbol == 'C']

            if len(nitrogens) == 2 and len(carbons) == Cn:
                vector = np.array([0.0, 0.0, 0.0])
                path = nx.shortest_path(G, source=nitrogens[0], target=nitrogens[1])
                for k in range(len(path) - 1):
                    diff = structure.get_distance(
                        path[k], path[k + 1], mic=True, vector=True
                    )
                    vector = vector + diff
                distances.append(np.linalg.norm(vector))

        return np.unique(np.array(distances).round(decimals=3))

    except Exception as e:
        print(f"Warning: Error reading {cif_file}: {e}. Skipping.")


def extract_code(filename):
    """Extract 6-letter CSD refcode from a CIF file path."""
    match = re.search(r'/([A-Z]{6})', filename)
    if match:
        return match.group(1)
    return None

In [ ]:
# Build dataframe of CIF files organized by chain length (Cn)
folders = [
    "c4_structures", "c5_structures", "c6_structures",
    "c7_structures", "c8_structures", "c9_structures",
    "c10_structures", "c11_structures", "c12_structures",
]
ns = [4, 5, 6, 7, 8, 9, 10, 11, 12]
schema = {"file": str, "Cn": int}

frames = []
for folder, n in zip(folders, ns):
    cif_files = glob.glob(os.path.join(folder, "*.cif"))
    cns = [n] * len(cif_files)
    frames.append(
        pd.DataFrame({"file": cif_files, "Cn": cns}).astype(schema)
    )

dfNN = pd.concat(frames, ignore_index=True)
dfNN["code"] = dfNN["file"].apply(extract_code)

In [ ]:
# Calculate N-N distances across all CIF files using all available CPU cores
def apply_distanceNN(row):
    return distanceNN(row["file"], row["Cn"])


dfNN["N-N distance"] = Parallel(n_jobs=-1)(
    delayed(apply_distanceNN)(row) for _, row in dfNN.iterrows()
)

In [ ]:
# Explode to one row per N-N distance measurement, drop missing values,
# remove the raw file path, and enforce numeric types
dfnn = dfNN.explode("N-N distance").dropna()
dfnn = dfnn.drop(columns=["file"])
dfnn["N-N distance"] = dfnn["N-N distance"].astype(float)
dfnn["code"] = dfnn["code"].astype(str)

In [ ]:
# Save exploded dataframe: one row per N-N distance measurement
dfnn.to_csv("analyzed_NN_exploded.csv")